Project Heading

**OUTLINE:**
1. Basic data inspection, preprocessing, and variable identification
- inspect dimensions
- calculate *wOBA*
2. Convert any categorical data to numeric as needed: pd.get_dummies()
- address null values if present
- remove outcomes from redictors to prevent data leakage (Hits, HRs, BBs)
3. Seaborn heatmap to highlight correlation between key variables like Exit Velocity and Launch Angle with the target var *wOBA*
- check for multicollinearity (calculate VIF)
4. Use sklearn for data training sets and to build linear regression model
- Train-test split
- Ordinary Least Squares Linear Regression
- Ridge or Lasso Regression (multicollinearity)
- Random Forest Regressor
- Cross-Validation for evaluaton reliability
5. Print formatted necessary data, including calculated R^2 and RMSE
- Compare R-squared and RMSE across models
- Extract feature importance
- Discuss limitations of the project

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

# machine learning / modeling
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

# metrics
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# stats
from statsmodels.stats.outliers_influence import variance_inflation_factor

# models
from sklearn.linear_model import RidgeCV, LassoCV

In [ ]:
url = "https://raw.githubusercontent.com/Hunter-Mott-31/2023_Baseball_Statcast_App/refs/heads/main/hitter_stats.csv"
df = pd.read_csv(url)
df.head()

,"last_name, first_name",player_id,year,home_run,batting_avg,slg_percent,on_base_percent,on_base_plus_slg,r_total_stolen_base,exit_velocity_avg,launch_angle_avg,whiff_percent,groundballs_percent,flyballs_percent,linedrives_percent,popups_percent,hp_to_1b,sprint_speed,Unnamed: 18
0,"Stephenson, Tyler",663886,2023,13,0.243,0.378,0.317,0.695,0,89.4,8.9,28.4,48.9,23.3,24.5,3.3,4.62,26.9,NaN
1,"Raleigh, Cal",663728,2023,30,0.232,0.456,0.306,0.762,0,89.5,20.3,29.9,31.7,35.1,22.2,11.0,4.56,27.0,NaN
2,"Diaz, Yandy",650490,2023,22,0.330,0.522,0.410,0.932,0,93.4,5.7,18.3,52.2,20.8,23.8,3.2,4.61,26.6,NaN
3,"Ramirez, Jose",608070,2023,24,0.282,0.475,0.356,0.831,28,90.0,18.0,15.6,34.9,28.5,26.3,10.3,4.38,27.8,NaN
4,"Smith, Dominic",642086,2023,12,0.254,0.366,0.326,0.692,1,86.3,12.8,19.9,42.7,25.8,26.3,5.3,4.56,25.8,NaN


*** Features should be raw physical metrics, such as exit velocity, launch angle, sprint speed, etc. ***

In [ ]:
### inspection
number_players, number_features = df.shape
print(f'Dataset contains {number_players} players and {number_features} features / columns.')

### drop NaN column(s)
df = df.drop(columns=['Unnamed: 18'])

Dataset contains 133 players and 19 features / columns.


In [ ]:
### assigning variables
# wOBA is not present in our dataset, and neither is the data required to manually calculate this.  On-Base Plus Slugging (OPS), however, is calculable from the
# data present, and is still a useful predictor.
y = df['on_base_plus_slg']

X = df[['exit_velocity_avg', 'launch_angle_avg', 'whiff_percent', 'groundballs_percent', 'flyballs_percent', 'linedrives_percent', 'popups_percent', 'sprint_speed']]
print(f'Shape of X: {X.shape}')
X.head()

Shape of X: (133, 8)


,exit_velocity_avg,launch_angle_avg,whiff_percent,groundballs_percent,flyballs_percent,linedrives_percent,popups_percent,sprint_speed
0,89.4,8.9,28.4,48.9,23.3,24.5,3.3,26.9
1,89.5,20.3,29.9,31.7,35.1,22.2,11.0,27.0
2,93.4,5.7,18.3,52.2,20.8,23.8,3.2,26.6
3,90.0,18.0,15.6,34.9,28.5,26.3,10.3,27.8
4,86.3,12.8,19.9,42.7,25.8,26.3,5.3,25.8


In [ ]:
### Checking multicollinearity - Brought up in Project Proposal feedback
X_vif = X.copy()
X_vif = X_vif.dropna()
X_vif_con = sm.add_constant(X_vif) # add constant column for statsmodel VIF

# VIF (Variance Inflation Factor) for each feature - This tells us the severity of multicollinearity between variables
vif_data = pd.DataFrame()
vif_data['Feature'] = X_vif.columns

vif_vals = []
for i in range(X_vif.shape[1]):
  # VIF calc
  vif_val = variance_inflation_factor(X_vif_con.values, i + 1)
  vif_vals.append(vif_val)

vif_data['VIF'] = vif_vals

print('~Variance Influence Factors~')
print('VIF of 1 means no correlation, 1-5 means moderate correlation, > 5 means high correlation.')
print(f'{vif_data.sort_values(by='VIF', ascending=False)}')

~Variance Influence Factors~
VIF of 1 means no correlation, 1-5 means moderate correlation, > 5 means high correlation.
               Feature           VIF
3  groundballs_percent  10174.957063
4     flyballs_percent   7040.692932
5   linedrives_percent   1889.593774
6       popups_percent   1814.537974
1     launch_angle_avg     23.023810
0    exit_velocity_avg      1.633409
2        whiff_percent      1.361480
7         sprint_speed      1.028990


5/8 values have a very high VIF score, meaning they will be immensely problematic for OLS (Ordinary Least Squares).  Cleanup is a requirement to continue:

In [ ]:
### Eliminating multicollinearity
# keeping distinct physical features
X_cleaned = df[['exit_velocity_avg', 'launch_angle_avg', 'whiff_percent', 'sprint_speed']].dropna()
X_cleaned_con = sm.add_constant(X_cleaned)

# VIF
vif_cleaned = pd.DataFrame()
vif_cleaned['Feature'] = X_cleaned.columns

vif_cleaned_vals = []
for i in range(X_cleaned.shape[1]):
  # VIF calc
  vif_cleaned_val = variance_inflation_factor(X_cleaned_con.values, i + 1)
  vif_cleaned_vals.append(vif_cleaned_val)

vif_cleaned['VIF'] = vif_cleaned_vals

print('~New Variance Influence Factors~')
print(f'{vif_cleaned.sort_values(by='VIF', ascending=False)}')

~New Variance Influence Factors~
             Feature       VIF
2      whiff_percent  1.234325
0  exit_velocity_avg  1.225390
1   launch_angle_avg  1.040839
3       sprint_speed  1.022008


In [ ]:
new_x = df[['exit_velocity_avg', 'launch_angle_avg', 'whiff_percent', 'sprint_speed']]
x_const = sm.add_constant(new_x)
model_sm = sm.OLS(y, x_const).fit()
model_sm.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:       on_base_plus_slg   R-squared:                       0.449
Model:                            OLS   Adj. R-squared:                  0.432
Method:                 Least Squares   F-statistic:                     26.07
Date:                Tue, 28 Jul 2026   Prob (F-statistic):           8.09e-16
Time:                        22:31:32   Log-Likelihood:                 184.39
No. Observations:                 133   AIC:                            -358.8
Df Residuals:                     128   BIC:                            -344.3
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
=====================================================================================
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                -1.8178      0.293     -6.205      0.000      -2.398      -1.238
exit_velocity_avg     0.0291      0.003      9.996      0.000       0.023       0.035
launch_angle_avg      0.0040      0.001      3.103      0.002       0.001       0.007
whiff_percent        -0.0043      0.001     -4.239      0.000      -0.006      -0.002
sprint_speed          0.0012      0.004      0.283      0.778      -0.007       0.010
==============================================================================
Omnibus:                        2.231   Durbin-Watson:                   1.879
Prob(Omnibus):                  0.328   Jarque-Bera (JB):                1.724
Skew:                           0.212   Prob(JB):                        0.422
Kurtosis:                       3.361   Cond. No.                     5.35e+03
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 5.35e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [ ]:
model_sk = LinearRegression()
model_sk.fit(x_const, y)

coef_df = pd.DataFrame({'feature': x_const.columns, 'coefficient': model_sk.coef_})
print('Intercept:', round(model_sk.intercept_, 2))
coef_df

Intercept: -1.82


,feature,coefficient
0,const,0.000000
1,exit_velocity_avg,0.029150
2,launch_angle_avg,0.004007
3,whiff_percent,-0.004280
4,sprint_speed,0.001232


In [ ]:
model_sm = sm.OLS(y, x_const).fit()
print(model_sm.summary())

# compute RMSE and R²
y_pred_sm = model_sm.fittedvalues
rmse_sm = np.sqrt(np.mean((y - y_pred_sm) ** 2))
r2_sm = model_sm.rsquared

                            OLS Regression Results                            
Dep. Variable:       on_base_plus_slg   R-squared:                       0.449
Model:                            OLS   Adj. R-squared:                  0.432
Method:                 Least Squares   F-statistic:                     26.07
Date:                Tue, 28 Jul 2026   Prob (F-statistic):           8.09e-16
Time:                        22:31:32   Log-Likelihood:                 184.39
No. Observations:                 133   AIC:                            -358.8
Df Residuals:                     128   BIC:                            -344.3
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                -1.8178      0.29

In [ ]:
next_x = df[['exit_velocity_avg', 'launch_angle_avg', 'whiff_percent']]
x_const = sm.add_constant(next_x)


model_sm = sm.OLS(y, x_const).fit()
print(model_sm.summary())

# compute RMSE and R²
y_pred_sm = model_sm.fittedvalues
rmse_sm = np.sqrt(np.mean((y - y_pred_sm) ** 2))
r2_sm = model_sm.rsquared

                            OLS Regression Results                            
Dep. Variable:       on_base_plus_slg   R-squared:                       0.449
Model:                            OLS   Adj. R-squared:                  0.436
Method:                 Least Squares   F-statistic:                     34.99
Date:                Tue, 28 Jul 2026   Prob (F-statistic):           1.30e-16
Time:                        22:33:14   Log-Likelihood:                 184.35
No. Observations:                 133   AIC:                            -360.7
Df Residuals:                     129   BIC:                            -349.1
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                -1.7761      0.25